In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib

In [ ]:
import torch
import numpy as np
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

from datasets import load_dataset, Audio, DatasetDict

### Φόρτωση συνόλου δεδομένων και speaker independent διαχωρισμός

In [ ]:
def load_savee_speaker_independent():
    """
    Συνάρτηση υπεύθυνη για τον speaker independent διαχωρισμό του SAVEE. Το σύνολο
    περιλαμβάνει 4 ομιλητές (DC, JE, JK, KL). Οι ομιλητές DC και JE αντιστοιχίζονται
    στο σύνολο εκπαίδευσης και οι JK και KL στο σύνολο δοκιμής
    """
    dataset = load_dataset("AbstractTTS/SAVEE", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        return example["file"].split("_")[0]

    train_speakers = ["DC", "JE"]
    test_speakers = ["JK", "KL"]

    train_dataset = dataset.filter(lambda x: get_speaker(x) in train_speakers, load_from_cache_file=False)
    test_dataset = dataset.filter(lambda x: get_speaker(x) in test_speakers, load_from_cache_file=False)

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_savee_speaker_independent()
print(dataset)

In [ ]:
#Εξαγωγή των ετικετών των συναισθημάτων σε λίστα
labels = sorted(set(dataset["train"]["emotion"]))
print(labels)

#Δημιουργία λεξικών αντιστοίχισης ετικετών και αριθμών
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = i
    id2label[i] = label

#υπολογισμός αριθμού ετικετών
num_labels = len(labels)
print(f"Διαθέσιμες ετικέτες: {labels}")
print(f"Αριθμός ετικετών: {num_labels}")

In [ ]:
model_repo = "openai/whisper-base"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_repo)

def prepare_dataset(batch):
    audio = batch["audio"]
    #Περικοπή στα 6 δευτερόλεπτα και μετατροπή σε log-mel spectrogram (το Whisper κάνει padding στα 30s)
    audio_array = audio["array"][:6 * 16000]
    batch["input_features"] = feature_extractor(
        audio_array, sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = label2id[batch["emotion"]]
    return batch

#Εφαρμογή της συνάρτησης σε όλο το dataset και αφαίρεση των υπόλοιπων στηλών
encoded_dataset = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, load_from_cache_file=False)

print("Encoded dataset features:")
print(encoded_dataset["train"].features)

max_abs_vals = [np.max(np.abs(x["input_features"])) for x in encoded_dataset["train"]]
print("\nΜέγιστη απόλυτη τιμή (train):", max(max_abs_vals))

###Ορισμός μετρικών αξιολόγησης

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,

    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,

    average_precision_score
)

from sklearn.preprocessing import label_binarize
from scipy.special import softmax

In [ ]:
#Λίστα για την αποθήκευση των confusion matrices
all_confusion_matrices = []

def compute_metrics(eval_pred):
    """
    Συνάρτηση για τον υπολογισμό των μετρικών ανά εποχή εκπαίδευσης, η οποία προσθέτει
    το confusion matrix κάθε εποχής στην all_confusion_matrices. Επειδή γίνεται εκ νέου
    υπολογισμός της καλύτερης εποχής μετά το τέλος της εκπαίδευσης, αυτή περιέχει ένα
    επιπλέον confusion matrix, ίδιο με το καλύτερο
     """
    logits, labels_true = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)

    labels_bin = label_binarize(labels_true, classes=list(range(num_labels)))

    # ROC-AUC (macro-averaged, one-vs-rest)
    try:
        roc_auc = roc_auc_score(labels_bin, probs, multi_class="ovr", average="macro")
    except ValueError:
        roc_auc = float("nan")

    # PR-AUC (macro-averaged average precision)
    try:
        pr_auc = average_precision_score(labels_bin, probs, average="macro")
    except ValueError:
        pr_auc = float("nan")

    accuracy = accuracy_score(labels_true, predictions)
    balanced_accuracy = balanced_accuracy_score(labels_true, predictions)
    precision = precision_score(labels_true, predictions, average="weighted", zero_division=0)
    recall = recall_score(labels_true, predictions, average="weighted", zero_division=0)
    f1 = f1_score(labels_true, predictions, average="weighted", zero_division=0)
    mcc = matthews_corrcoef(labels_true, predictions)
    k_score = cohen_kappa_score(labels_true, predictions)
    logloss = log_loss(labels_true, probs, labels=list(range(num_labels)))

    cm = confusion_matrix(labels_true, predictions, labels=list(range(num_labels)))
    all_confusion_matrices.append(cm)

    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc" : mcc,
        "k_score" : k_score,
        "log_loss": logloss,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    }

### Φόρτωση του αρχικού προεκπαιδευμένου μοντέλου

In [ ]:
from transformers import WhisperForAudioClassification

model = WhisperForAudioClassification.from_pretrained(
    model_repo, num_labels=num_labels, label2id=label2id, id2label=id2label,
    ignore_mismatched_sizes=True,
).to("cuda")

if torch.cuda.is_available():
    model = model.to('cuda')
print(f"Το μοντέλο φορτώθηκε στη συσκευή: {model.device}")

### Ορισμός παραμέτρων εκπαίδευσης

In [ ]:
#Δυναμικός υπολογισμός των warmup_steps (10% των συνολικών βημάτων εκπαίδευσης),
per_device_train_batch_size = 8
gradient_accumulation_steps = 4
num_train_epochs = 15

effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = max(1, len(encoded_dataset["train"]) // effective_batch_size)
total_steps = steps_per_epoch * num_train_epochs
warmup_steps = max(1, int(0.1 * total_steps))

#Καθορισμός μετρικών εκπαίδευσης του μοντέλου
training_args = TrainingArguments(
    output_dir="./whisper-finetuned-SAVEE",
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=gradient_accumulation_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=num_train_epochs,
    fp16=False,
    learning_rate=3e-5,
    warmup_steps=warmup_steps,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    push_to_hub=False,
    report_to="none",
)

###Cell 4 — DataCollator (προσαρμοσμένο στο input_features)

In [ ]:
import torch

class DataCollatorWhisperClassification:
    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor

    def __call__(self, features):
        input_features = torch.stack([torch.tensor(f["input_features"]) for f in features])
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return {"input_features": input_features, "labels": labels}

data_collator = DataCollatorWhisperClassification(feature_extractor)

In [ ]:
all_confusion_matrices.clear()

### Δημιουργία και Εκπαίδευση του Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Εκκίνηση εκπαίδευσης...")
trainer.train()
print("Εκπαίδευση ολοκληρώθηκε!")

In [ ]:
#Εκτύπωση των τελικών αποτελεσμάτων της εκπαίδευσης
eval_results = trainer.evaluate()
print(f"Αποτελέσματα αξιολόγησης: {eval_results}")

### Οπτικοποίηση αποτελεσμάτων (Confusion Matrices)

In [ ]:
#Αποθήκευση των ετικετών για τους άξονες των matrices
labels = [
    model.config.id2label[i]
    for i in sorted(model.config.id2label.keys())
]

#Loop στην λίστα με τα confusion matrices
for epoch, cm in enumerate(all_confusion_matrices):
  #Δημιουργία γραφήματος
  plt.figure(figsize=(6,5))

  #Δημιουργία Heatmap
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=labels,
      yticklabels=labels
  )

  plt.title(f"Confusion Matrix - Epoch {epoch+1}")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.show()

### Αποθήκευση εκπαιδευμένου μοντέλου στο HuggingFace Hub

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [ ]:
repo_id = "VasilisAsim/whisper-finetuned-SAVEE"

#Αποθήκευση του μοντέλου στο HuggingFace
model.push_to_hub(repo_id)

#Αποθήκευση του Feature Extractor
feature_extractor.push_to_hub(repo_id)

print(f"Το μοντέλο και ο επεξεργαστής αποθηκεύτηκαν στο Hugging Face Hub με ID: {repo_id}")